In [ ]:
# 1. INSTALLS & IMPORTS
# !pip install -q transformers datasets scikit-learn peft wandb
# !pip install -q git+https://github.com/huggingface/transformers.git

import pandas as pd
import numpy as np
import os
import torch
import random
import shutil
from datasets import Dataset, Value
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import wandb
# from google.colab import drive

# drive.mount('/content/drive')
wandb.login()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 2. CONFIGURATION
RUN_NAME      = "model_w_fd_d02_09"
LEARNING_RATE = 2e-4
LORA_RANK     = 32
NUM_EPOCHS    = 6
BATCH_SIZE    = 16

# 3. DATA PREPARATION (ALL DATA)
df = pd.read_csv('combined02.csv')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# NEW: build richer text with followers encoded
# your df["text"] already starts with "Twitter: ..." / "LinkedIn: ..."
df["input_text"] = (  " | Followers: " + df["followers"].astype(str) + " | Year: " + df["year"].astype(str) + "| Platform: " + df["platform"].astype(str) + " | Followers: " + df["followers"].astype(str) + " | Year: " + df["year"].astype(str) + df["text"]   )

hf = Dataset.from_pandas(df[["input_text", "likes"]])
hf = hf.train_test_split(test_size=0.2, seed=42)

tokenizer = AutoTokenizer.from_pretrained('answerdotai/ModernBERT-base')

def preprocess(examples):
    # tokenize the new combined field
    return tokenizer(
        examples['input_text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

def to_log_labels(batch):
    batch["labels"] = [float(np.log1p(x)) for x in batch["likes"]]
    return batch

hf = hf.map(preprocess, batched=True)
hf = hf.map(to_log_labels, batched=True)
hf = hf.remove_columns(["input_text", "likes"])
hf = hf.cast_column("labels", Value("float32"))
hf.set_format("torch")

# 4. MODEL SETUP
model = AutoModelForSequenceClassification.from_pretrained(
    'answerdotai/ModernBERT-base', num_labels=1
)
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=["attn.Wqkv", "attn.Wo", "mlp.Wi", "mlp.Wo"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
model = get_peft_model(model, lora_config)

# 5. METRICS (LOG SPACE)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.squeeze()
    rmse_log = np.sqrt(mean_squared_error(labels, preds))
    mae_log = mean_absolute_error(labels, preds)
    r2_log = r2_score(labels, preds)
    return {"rmse_log": rmse_log, "mae_log": mae_log, "r2_log": r2_log}

# 6. TRAIN
training_args = TrainingArguments(
    output_dir=f"./{RUN_NAME}",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    fp16=True,
    load_best_model_at_end=True,
    weight_decay=0.01,
    report_to="wandb",
    run_name=RUN_NAME,
    logging_steps=10,
    dataloader_num_workers=2,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf["train"],
    eval_dataset=hf["test"],
    compute_metrics=compute_metrics,
)

print(f"Starting Combined Training: {RUN_NAME}...")
trainer.train()


wandb: Currently logged in as: shaninee (shaninee-czech-technical-university-in-prague) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/1534 [00:00<?, ? examples/s]

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Map:   0%|          | 0/1534 [00:00<?, ? examples/s]

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

...

Casting the dataset:   0%|          | 0/384 [00:00<?, ? examples/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting Combined Training: model_w_fd_d02_09...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
# from google.colab import drive, files

# # --- 7. SAVE TO DRIVE & DOWNLOAD ---

# # A. Mount Google Drive
# drive.mount('/content/drive')

# # Define Paths
# local_model_path = f"./{RUN_NAME}"  # Where Trainer saved it temporarily
# drive_save_path  = f"/content/drive/My Drive/ModernBERT_Project/{RUN_NAME}"
# zip_filename     = f"{RUN_NAME}.zip"

# print(f"Saving model temporarily to: {local_model_path}")
# trainer.save_model(local_model_path)
# tokenizer.save_pretrained(local_model_path)

# # B. Copy to Google Drive
# print(f"Copying to Google Drive: {drive_save_path}...")
# if os.path.exists(drive_save_path):
#     shutil.rmtree(drive_save_path) # Clear old version if exists
# shutil.copytree(local_model_path, drive_save_path)
# print("✅ Saved to Google Drive!")

# # C. Download to Local Computer
# print("Zipping and downloading to your computer...")
# shutil.make_archive(RUN_NAME, 'zip', local_model_path) # Create zip file
# files.download(zip_filename) # Trigger download
# print("✅ Download started!")

In [ ]:
# 8. PREVIEW RESULTS (REAL NUMBERS)
preds = trainer.predict(hf["test"]).predictions.squeeze()
labels = trainer.predict(hf["test"]).label_ids
results = pd.DataFrame({
    "True": np.expm1(labels),
    "Pred": np.maximum(0, np.expm1(preds))
})
results["Diff"] = abs(results["True"] - results["Pred"])
print(results.head(10))
print("MAE (Real):", results["Diff"].mean())

wandb.finish()